# Exploration des données du Pipeline ETL

Ce notebook permet d'explorer les données stockées dans MinIO (Bronze, Silver, Gold).

In [ ]:
# Installation des dépendances si nécessaire
# !pip install minio pandas pyarrow

In [ ]:
import pandas as pd
import io
from minio import Minio

# Connexion à MinIO
client = Minio(
    endpoint="localhost:9000",
    access_key="minioadmin",
    secret_key="minioadmin123",
    secure=False
)

print("Buckets disponibles :")
for bucket in client.list_buckets():
    print(f"  - {bucket.name}")

In [ ]:
def load_latest_parquet(bucket: str, prefix: str) -> pd.DataFrame:
    """Charge le fichier Parquet le plus récent depuis MinIO."""
    objects = list(client.list_objects(bucket, prefix=prefix, recursive=True))
    if not objects:
        raise FileNotFoundError(f"Aucun fichier dans {bucket}/{prefix}")
    
    latest = sorted(objects, key=lambda x: x.object_name)[-1]
    print(f"Chargement de : {bucket}/{latest.object_name}")
    
    response = client.get_object(bucket, latest.object_name)
    return pd.read_parquet(io.BytesIO(response.read()))

## Bronze Layer — Données brutes

In [ ]:
# Livres scrapés depuis books.toscrape.com
df_books_raw = load_latest_parquet("bronze", "books_web")
print(f"Shape: {df_books_raw.shape}")
df_books_raw.head()

In [ ]:
# Données météo brutes
df_weather_raw = load_latest_parquet("bronze", "weather")
print(f"Shape: {df_weather_raw.shape}")
df_weather_raw.head()

## Silver Layer — Données nettoyées

In [ ]:
df_books_clean = load_latest_parquet("silver", "books_web")
print(f"Shape après nettoyage: {df_books_clean.shape}")
df_books_clean.describe()

## Gold Layer — Agrégations

In [ ]:
# KPIs par catégorie
df_by_category = load_latest_parquet("gold", "books_by_category")
df_by_category.sort_values("nb_books", ascending=False).head(10)

In [ ]:
# Top livres les mieux notés
df_top = load_latest_parquet("gold", "books_top_rated")
df_top

In [ ]:
# Météo journalière
df_weather_daily = load_latest_parquet("gold", "weather_daily")
df_weather_daily